# Notebook 03: Multi-Label Classifier Fine-Tuning

**Project:** Hotel Curator -- Multi-Attribute Hotel Quality Classifier

**Inputs (from NB01, NB02a, NB02b):**
- `data/processed/reviews_clean.parquet` -- 20,475 cleaned reviews
- `data/processed/silver_labels_augmented.parquet` -- regex + LLM labels
- `data/processed/attribute_taxonomy.json` -- label schema
- `data/splits/hotel_split_mapping.json` -- hotel-level splits

**Objective:** Fine-tune `roberta-base` for multi-label classification
on the 16 `(attribute, polarity)` heads. Use BCE-with-logits with
inverse-frequency `pos_weight` to address severe class imbalance.
Tune per-label decision thresholds on validation. Report per-label
test F1 with explicit caveats for low-prevalence labels.

**Alignment with `ML Implementation Plan - Hotel Curator.pdf`:**

This notebook implements **Phase 5** (MLM domain adaptation) and
**Phase 6** (multi-label classification fine-tuning) from the spec.

Documented deviations from spec defaults:
1. **`max_seq_length=256` (not 128)** -- corpus p95 is 275 words;
   128 would truncate ~50% of reviews mid-content.
2. **`pos_weight` uses sqrt strategy, capped at 50x** -- raw inverse
   frequency for `wifi_negative` would be ~290x and gradient-unstable.

---

## Pipeline

0. Setup -- imports, config, seed (random + numpy + torch + cuda)
1. Load artifacts; build train/val/test from hotel splits
2. **MLM domain adaptation** -- 3 epoch of masked-LM on training
   reviews to adapt `roberta-base` to hospitality jargon (Phase 5)
3. Build HuggingFace `Dataset` objects + tokenization
4. Compute per-label `pos_weight` from training set
5. Model setup with multi-label problem type (from MLM checkpoint)
6. Custom `WeightedBCETrainer` (subclasses HF `Trainer`)
7. Compute multi-label F1 metrics (micro / macro / per-label)
8. Training arguments and run
9. Per-label threshold tuning on validation
10. Test set evaluation with thresholded predictions
11. Persist model + thresholds + label names + metrics

## Outputs

| Artifact | Description |
|---|---|
| `checkpoints/roberta-tripadvisor-mlm/` | Domain-adapted MLM checkpoint (Phase 5) |
| `checkpoints/hotel_curator_clf/` | Trained classifier + tokenizer (HF format) |
| `checkpoints/hotel_curator_clf/thresholds.json` | Per-label decision thresholds |
| `checkpoints/hotel_curator_clf/label_names.json` | Ordered list of 16 labels |
| `checkpoints/hotel_curator_clf/metrics.json` | Val + test F1 (micro/macro/per-label) |
| `checkpoints/hotel_curator_clf/training_log.json` | Step-wise loss, eval F1 |

In [1]:
# --- 0.0 Install dependencies (run once per environment) ---
# Already done if you followed env setup. Uncomment if needed.

# %pip install transformers datasets torch scikit-learn accelerate wandb

In [2]:
# --- 0.1 Imports ---

import json
import os
import random
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from sklearn.metrics import (
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from torch import nn
from transformers import (
    AutoModelForMaskedLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    DataCollatorWithPadding,
    EvalPrediction,
    Trainer,
    TrainingArguments,
)

# Silence noisy warnings on Windows
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [3]:
# --- 0.2 Configuration ---

SEED = 42

# Paths -- inputs
DATA_DIR = Path("data")
PROCESSED_DIR = DATA_DIR / "processed"
SPLITS_DIR = DATA_DIR / "splits"

REVIEWS_PATH = PROCESSED_DIR / "reviews_clean.parquet"
SILVER_PATH = PROCESSED_DIR / "silver_labels_augmented.parquet"
TAXONOMY_PATH = PROCESSED_DIR / "attribute_taxonomy.json"
SPLIT_PATH = SPLITS_DIR / "hotel_split_mapping.json"

# Paths -- outputs
CKPT_DIR = Path("checkpoints/hotel_curator_clf")
MLM_CKPT_DIR = Path("checkpoints/roberta-tripadvisor-mlm")
TRAINER_OUTPUT_DIR = Path("checkpoints/_trainer_runs")

# Base model
MODEL_NAME = "roberta-base"

# --- Phase 5: MLM domain adaptation ---
RUN_MLM_ADAPTATION = False       # Phase 5 from ML Implementation Plan
MLM_MAX_SEQ_LENGTH = 128
MLM_BATCH_SIZE = 64             # was 32; GPU has headroom at seq 128
MLM_EPOCHS = 3                  # was 1; small corpus needs more passes
MLM_LEARNING_RATE = 3e-5        # was 5e-5; lower peak for 3x more steps
MLM_PROBABILITY = 0.15

# --- Phase 6: Classification fine-tuning ---
MAX_SEQ_LENGTH = 256            # p95 review is 275 words; 128 too short
BATCH_SIZE = 64                 # RTX 5090 has the VRAM
LEARNING_RATE = 2e-5
NUM_EPOCHS = 5
WARMUP_RATIO = 0.06
WEIGHT_DECAY = 0.01

# Class imbalance handling
POS_WEIGHT_STRATEGY = "sqrt"    # sqrt softens extremes vs hard cap
POS_WEIGHT_CAP = 50.0           # safety bound

# Eval / save / early stop
EVAL_STEPS = 100
SAVE_TOTAL_LIMIT = 2
METRIC_FOR_BEST = "macro_f1"
EARLY_STOPPING_PATIENCE = 3

# Optional: Weights & Biases (Phase 6 spec: "Log metrics to W&B")
# Set True if wandb is installed and WANDB_API_KEY is configured.
USE_WANDB = False

# Reduced run for quick sanity checks. Set False for full training.
QUICK_TEST = False
QUICK_TEST_TRAIN_N = 1_000
QUICK_TEST_VAL_N = 200
QUICK_TEST_EPOCHS = 1

print("Config loaded.")
print(f"  MODEL_NAME          = {MODEL_NAME}")
print(f"  RUN_MLM_ADAPTATION  = {RUN_MLM_ADAPTATION}")
print(f"  MAX_SEQ_LENGTH      = {MAX_SEQ_LENGTH}")
print(f"  BATCH_SIZE          = {BATCH_SIZE}")
print(f"  NUM_EPOCHS          = {NUM_EPOCHS}")
print(f"  POS_WEIGHT_CAP      = {POS_WEIGHT_CAP}")
print(f"  USE_WANDB           = {USE_WANDB}")
print(f"  QUICK_TEST          = {QUICK_TEST}")

Config loaded.
  MODEL_NAME          = roberta-base
  RUN_MLM_ADAPTATION  = False
  MAX_SEQ_LENGTH      = 256
  BATCH_SIZE          = 64
  NUM_EPOCHS          = 5
  POS_WEIGHT_CAP      = 50.0
  USE_WANDB           = False
  QUICK_TEST          = False


In [4]:
# --- 0.3 Reproducibility + device detection ---


def set_seed(seed: int) -> None:
    """Set seeds across random, numpy, torch, and CUDA."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Seed:     {SEED}")
print(f"Device:   {device}")
if device == "cuda":
    print(f"GPU:      {torch.cuda.get_device_name(0)}")
    print(
        f"VRAM:     "
        f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB"
    )

Seed:     42
Device:   cuda
GPU:      NVIDIA GeForce RTX 5090 Laptop GPU
VRAM:     25.7 GB


---

## Section 1: Build Train/Val/Test from Hotel Splits

Load the augmented silver labels and join to review text. Apply
the hotel-level split mapping from NB02a so reviews from the same
hotel never cross the train/val/test boundary.

In [5]:
# --- 1.1 Load artifacts and join ---


def load_artifacts(
    reviews_path: Path,
    silver_path: Path,
    taxonomy_path: Path,
    split_path: Path,
) -> tuple[pd.DataFrame, list[str], dict[str, list[int]]]:
    """Load reviews + silver labels + taxonomy + splits, join."""
    for p in (reviews_path, silver_path, taxonomy_path, split_path):
        if not p.exists():
            raise FileNotFoundError(
                f"{p} missing. Run prior notebooks first."
            )
    df_reviews = pd.read_parquet(reviews_path)
    df_silver = pd.read_parquet(silver_path)
    with taxonomy_path.open() as f:
        taxonomy = json.load(f)
    with split_path.open() as f:
        splits = json.load(f)

    label_cols = list(taxonomy["labels"])
    df = df_reviews.merge(df_silver, on="review_id", how="inner")
    expected_cols = ["review_id", "hotel_id", "text", "rating", *label_cols]
    missing = set(expected_cols) - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns after join: {missing}")
    return df[expected_cols], label_cols, splits


df, ALL_LABELS, hotel_splits = load_artifacts(
    REVIEWS_PATH, SILVER_PATH, TAXONOMY_PATH, SPLIT_PATH,
)
NUM_LABELS = len(ALL_LABELS)

print(f"Joined dataset: {df.shape}")
print(f"Labels: {NUM_LABELS}")
print(f"Hotels: train={len(hotel_splits['train'])}, "
      f"val={len(hotel_splits['val'])}, "
      f"test={len(hotel_splits['test'])}")

Joined dataset: (20475, 20)
Labels: 16
Hotels: train=160, val=20, test=20


In [6]:
# --- 1.2 Apply hotel-level split ---


def apply_hotel_split(
    df: pd.DataFrame,
    splits: dict[str, list[int]],
) -> dict[str, pd.DataFrame]:
    """Partition reviews by hotel_id into train/val/test."""
    out: dict[str, pd.DataFrame] = {}
    for split_name, hotel_ids in splits.items():
        mask = df["hotel_id"].isin(hotel_ids)
        out[split_name] = df[mask].reset_index(drop=True)
    return out


split_dfs = apply_hotel_split(df, hotel_splits)

for name, sdf in split_dfs.items():
    print(f"  {name:5s}: {len(sdf):>6,} reviews "
          f"({sdf['hotel_id'].nunique()} hotels)")

print("\nPer-split label prevalence (% positive):")
header = f"  {'label':<26s} " + " ".join(
    [f"{n:>7s}" for n in ("train", "val", "test")]
)
print(header)
print(f"  {'-' * 26} {'-' * 7} {'-' * 7} {'-' * 7}")
for lbl in ALL_LABELS:
    rates = [
        split_dfs[name][lbl].mean() * 100
        for name in ("train", "val", "test")
    ]
    print(f"  {lbl:<26s} {rates[0]:>6.2f}% "
          f"{rates[1]:>6.2f}% {rates[2]:>6.2f}%")

  train: 16,412 reviews (160 hotels)
  val  :  2,046 reviews (20 hotels)
  test :  2,017 reviews (20 hotels)

Per-split label prevalence (% positive):
  label                        train     val    test
  -------------------------- ------- ------- -------
  cleanliness_positive        39.37%  38.12%  38.77%
  cleanliness_negative         7.09%   5.52%   5.65%
  room_positive               41.84%  35.92%  36.99%
  room_negative               12.89%   9.38%  10.31%
  staff_positive              53.35%  48.09%  46.50%
  staff_negative               7.57%   4.84%   3.82%
  location_positive           36.74%  33.14%  32.23%
  location_negative            2.50%   1.22%   1.34%
  noise_positive              13.25%  13.34%  14.03%
  noise_negative               8.02%   7.23%   8.87%
  wifi_positive                3.16%   2.83%   2.78%
  wifi_negative                0.38%   0.20%   0.15%
  value_positive              14.29%  11.73%  10.11%
  value_negative               6.99%   4.06%   4.16%
 

In [7]:
# --- 1.3 Optional quick-test downsample ---

if QUICK_TEST:
    rng = np.random.default_rng(SEED)
    sampled = {}
    for name, sdf in split_dfs.items():
        n = QUICK_TEST_TRAIN_N if name == "train" else QUICK_TEST_VAL_N
        n = min(n, len(sdf))
        idx = rng.choice(len(sdf), size=n, replace=False)
        sampled[name] = sdf.iloc[idx].reset_index(drop=True)
    split_dfs = sampled
    print("QUICK_TEST: subsampled splits")
    for name, sdf in split_dfs.items():
        print(f"  {name:5s}: {len(sdf):>6,} reviews")
else:
    print("QUICK_TEST = False (using full splits)")

QUICK_TEST = False (using full splits)


---

## Section 2: MLM Domain Adaptation (Phase 5)

We use the training reviews (~16K documents) as the MLM corpus.
The plan recommends sentence-level input, but our corpus lacks
reliable sentence boundaries (lowercased, missing terminal
punctuation -- see NB01 Section 4). Using full reviews at
`max_seq_length=128` means each review contributes 1-3 chunks
after truncation, which is comparable coverage.

The adapted checkpoint is saved to `checkpoints/roberta-tripadvisor-mlm/`
and loaded as the base for classification fine-tuning in Section 5.

**Skip behavior:** Set `RUN_MLM_ADAPTATION = False` in config to
skip this section and fine-tune directly from `roberta-base` (the
original NB03 behavior).

In [8]:
# --- 2.1 Prepare MLM corpus from training reviews ---

if RUN_MLM_ADAPTATION:
    mlm_texts = split_dfs["train"]["text"].tolist()
    print(f"MLM corpus: {len(mlm_texts):,} training reviews")
    print(f"  total chars: {sum(len(t) for t in mlm_texts):,}")
    mlm_dataset = Dataset.from_dict({"text": mlm_texts})
    print(f"  HF Dataset:  {mlm_dataset}")
else:
    print("RUN_MLM_ADAPTATION = False -- skipping MLM phase.")

RUN_MLM_ADAPTATION = False -- skipping MLM phase.


In [9]:
# --- 2.2 Tokenize for MLM ---

# Load tokenizer once -- shared between MLM and classification.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if RUN_MLM_ADAPTATION:
    def _mlm_tokenize(batch: dict[str, list]) -> dict[str, list]:
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=MLM_MAX_SEQ_LENGTH,
            padding=False,
        )

    mlm_tokenized = mlm_dataset.map(
        _mlm_tokenize,
        batched=True,
        remove_columns=["text"],
        desc="Tokenizing (MLM)",
    )
    mlm_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=True,
        mlm_probability=MLM_PROBABILITY,
    )
    print(f"MLM tokenized: {mlm_tokenized}")
    print(f"  masking probability: {MLM_PROBABILITY}")

In [10]:
# --- 2.3 Run MLM training (3 epoch) ---

if RUN_MLM_ADAPTATION:
    MLM_CKPT_DIR.mkdir(parents=True, exist_ok=True)

    mlm_model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)
    n_mlm_params = sum(p.numel() for p in mlm_model.parameters())
    print(f"MLM model: {MODEL_NAME} ({n_mlm_params / 1e6:.1f}M params)")

    mlm_args = TrainingArguments(
        output_dir=str(MLM_CKPT_DIR / "_runs"),
        num_train_epochs=MLM_EPOCHS,
        per_device_train_batch_size=MLM_BATCH_SIZE,
        learning_rate=MLM_LEARNING_RATE,
        weight_decay=0.01,
        warmup_ratio=0.06,
        save_strategy="no",
        logging_steps=50,
        bf16=(device == "cuda"),
        fp16=False,
        dataloader_num_workers=0,
        report_to=["wandb"] if USE_WANDB else [],
        seed=SEED,
    )

    mlm_trainer = Trainer(
        model=mlm_model,
        args=mlm_args,
        train_dataset=mlm_tokenized,
        data_collator=mlm_collator,
    )

    t0 = time.time()
    print(f"\nStarting MLM training ({MLM_EPOCHS} epoch)...")
    mlm_result = mlm_trainer.train()
    mlm_wall = time.time() - t0

    print(f"\nMLM training done.")
    print(f"  wall time:   {mlm_wall:.1f} s")
    print(f"  final loss:  {mlm_result.metrics['train_loss']:.4f}")
    print(f"  steps/sec:   {mlm_result.metrics['train_steps_per_second']:.2f}")

In [11]:
# --- 2.4 Save MLM checkpoint ---

if RUN_MLM_ADAPTATION:
    mlm_model.save_pretrained(str(MLM_CKPT_DIR))
    tokenizer.save_pretrained(str(MLM_CKPT_DIR))

    mlm_meta = {
        "base_model": MODEL_NAME,
        "mlm_epochs": MLM_EPOCHS,
        "mlm_max_seq_length": MLM_MAX_SEQ_LENGTH,
        "mlm_batch_size": MLM_BATCH_SIZE,
        "mlm_learning_rate": MLM_LEARNING_RATE,
        "mlm_probability": MLM_PROBABILITY,
        "corpus_size": len(mlm_texts),
        "final_loss": float(mlm_result.metrics["train_loss"]),
        "wall_time_s": round(mlm_wall, 1),
    }
    with (MLM_CKPT_DIR / "mlm_meta.json").open("w") as f:
        json.dump(mlm_meta, f, indent=2)

    print(f"MLM checkpoint saved to: {MLM_CKPT_DIR}")
    for p in sorted(MLM_CKPT_DIR.iterdir()):
        if p.is_file():
            print(f"  {p.name}  ({p.stat().st_size / 1e6:.2f} MB)")

    # Free MLM model from GPU before classification phase
    del mlm_model, mlm_trainer
    if device == "cuda":
        torch.cuda.empty_cache()
    print("\nMLM model freed from memory.")

# Resolve the base model for classification fine-tuning
CLF_BASE_MODEL = str(MLM_CKPT_DIR) if RUN_MLM_ADAPTATION else MODEL_NAME
print(f"\nClassification base model: {CLF_BASE_MODEL}")


Classification base model: roberta-base


---

## Section 3: Build HuggingFace `Dataset` Objects

Each split is converted to a `Dataset` with columns:
- `text` -- the review string
- `labels` -- a 16-float list (multi-hot)

`Trainer` expects `labels` as a single column even for multi-label
problems. We pre-pack the 16 binary columns into one list per row.

In [12]:
# --- 3.1 Build HuggingFace DatasetDict ---


def to_hf_dataset(
    df: pd.DataFrame,
    label_cols: list[str],
) -> Dataset:
    """Convert a DataFrame to HF Dataset with text and labels."""
    out = pd.DataFrame({
        "text": df["text"].astype(str),
        "labels": df[label_cols].astype(np.float32).values.tolist(),
    })
    return Dataset.from_pandas(out, preserve_index=False)


datasets = DatasetDict({
    "train": to_hf_dataset(split_dfs["train"], ALL_LABELS),
    "validation": to_hf_dataset(split_dfs["val"], ALL_LABELS),
    "test": to_hf_dataset(split_dfs["test"], ALL_LABELS),
})

print(datasets)
print()
print("Sample row (train[0]):")
print(f"  text:   {datasets['train'][0]['text'][:120]}...")
print(f"  labels: {datasets['train'][0]['labels']}")

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 16412
    })
    validation: Dataset({
        features: ['text', 'labels'],
        num_rows: 2046
    })
    test: Dataset({
        features: ['text', 'labels'],
        num_rows: 2017
    })
})

Sample row (train[0]):
  text:   nice hotel expensive parking got good deal stay hotel anniversary, arrived late evening took advice previous reviews did...
  labels: [1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0]


In [13]:
# --- 3.2 Tokenization ---


def build_tokenize_fn(tok, max_length: int):
    """Closure capturing the tokenizer + max_length."""
    def _tokenize(batch: dict[str, list]) -> dict[str, list]:
        return tok(
            batch["text"],
            truncation=True,
            max_length=max_length,
            padding=False,
        )
    return _tokenize


tokenize_fn = build_tokenize_fn(tokenizer, MAX_SEQ_LENGTH)
tokenized = datasets.map(
    tokenize_fn, batched=True,
    remove_columns=["text"], desc="Tokenizing",
)
print(tokenized)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Tokenizing:   0%|          | 0/16412 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/2046 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/2017 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 16412
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 2046
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 2017
    })
})


---

## Section 4: Compute `pos_weight` for BCE Loss

For `BCEWithLogitsLoss`, `pos_weight` rescales the positive class
loss per label. We use a **sqrt** strategy: `min(sqrt(n_neg/n_pos), 50)`.
This preserves rank ordering between rare labels while preventing
gradient instability from extreme weights (~290x for `wifi_negative`).

In [14]:
# --- 4.1 Per-label pos_weight tensor ---


def compute_pos_weights(
    df: pd.DataFrame,
    label_cols: list[str],
    *,
    strategy: str = "sqrt",
    cap: float = 50.0,
) -> torch.Tensor:
    """Inverse-frequency pos_weight per label.

    Strategies:
        "cap":  min(n_neg / n_pos, cap)
        "sqrt": min(sqrt(n_neg / n_pos), cap)
    """
    if strategy not in {"cap", "sqrt"}:
        raise ValueError(f"Unknown strategy: {strategy}")
    weights = []
    for lbl in label_cols:
        n_pos = float(df[lbl].sum())
        n_neg = float(len(df) - n_pos)
        if n_pos == 0:
            w = cap
        else:
            raw = n_neg / n_pos
            if strategy == "sqrt":
                raw = float(np.sqrt(raw))
            w = min(raw, cap)
        weights.append(w)
    return torch.tensor(weights, dtype=torch.float32)


pos_weight = compute_pos_weights(
    split_dfs["train"], ALL_LABELS,
    strategy=POS_WEIGHT_STRATEGY, cap=POS_WEIGHT_CAP,
)
print(f"pos_weight strategy: {POS_WEIGHT_STRATEGY}")
print(f"pos_weight tensor shape: {pos_weight.shape}")
print()
print(f"  {'label':<26s} {'n_pos':>7s} {'pos_weight':>12s}")
print(f"  {'-' * 26} {'-' * 7} {'-' * 12}")
for lbl, w in zip(ALL_LABELS, pos_weight.tolist(), strict=True):
    n_pos = int(split_dfs["train"][lbl].sum())
    print(f"  {lbl:<26s} {n_pos:>7,} {w:>12.2f}")

pos_weight strategy: sqrt
pos_weight tensor shape: torch.Size([16])

  label                        n_pos   pos_weight
  -------------------------- ------- ------------
  cleanliness_positive         6,461         1.24
  cleanliness_negative         1,164         3.62
  room_positive                6,866         1.18
  room_negative                2,115         2.60
  staff_positive               8,756         0.94
  staff_negative               1,243         3.49
  location_positive            6,030         1.31
  location_negative              411         6.24
  noise_positive               2,175         2.56
  noise_negative               1,316         3.39
  wifi_positive                  519         5.53
  wifi_negative                   63        16.11
  value_positive               2,345         2.45
  value_negative               1,147         3.65
  breakfast_positive             891         4.17
  breakfast_negative             289         7.47


---

## Section 5: Model and Custom Trainer

**Phase 5 integration:** When `RUN_MLM_ADAPTATION = True`, the model
is loaded from the domain-adapted `roberta-tripadvisor-mlm` checkpoint.
The MLM head is discarded and replaced with the classification head --
standard HuggingFace behavior when loading `ForMaskedLM` into
`ForSequenceClassification`.

To inject `pos_weight`, we subclass `Trainer` and override
`compute_loss`.

In [15]:
# --- 5.1 Build model and custom trainer ---


def build_model(
    model_name_or_path: str,
    num_labels: int,
) -> AutoModelForSequenceClassification:
    """Load a pre-trained model with a multi-label head.

    When loading from the MLM-adapted checkpoint, the LM head is
    discarded and replaced with a classification head.
    """
    return AutoModelForSequenceClassification.from_pretrained(
        model_name_or_path,
        num_labels=num_labels,
        problem_type="multi_label_classification",
    )


class WeightedBCETrainer(Trainer):
    """Trainer subclass that applies per-label pos_weight."""

    def __init__(self, *args, pos_weight: torch.Tensor, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.BCEWithLogitsLoss(
            pos_weight=self.pos_weight.to(logits.device)
        )
        loss = loss_fct(logits, labels.float())
        return (loss, outputs) if return_outputs else loss


model = build_model(CLF_BASE_MODEL, NUM_LABELS)
n_params = sum(p.numel() for p in model.parameters())
print(f"Classification model loaded from: {CLF_BASE_MODEL}")
print(f"Trainable params: {n_params / 1e6:.1f}M")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Classification model loaded from: roberta-base
Trainable params: 124.7M


---

## Section 6: Multi-Label F1 Metrics

`compute_metrics` is called at every evaluation pass. It applies
sigmoid + 0.5 threshold and reports micro, macro, and weighted F1.
We track `macro_f1` as the best-checkpoint metric.

In [16]:
# --- 6.1 Multi-label F1 metrics ---


def compute_metrics(eval_pred: EvalPrediction) -> dict[str, float]:
    """Multi-label F1 at training-time threshold = 0.5."""
    logits = eval_pred.predictions
    labels = eval_pred.label_ids
    probs = 1.0 / (1.0 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    labels_int = labels.astype(int)
    return {
        "micro_f1": float(f1_score(
            labels_int, preds, average="micro", zero_division=0,
        )),
        "macro_f1": float(f1_score(
            labels_int, preds, average="macro", zero_division=0,
        )),
        "weighted_f1": float(f1_score(
            labels_int, preds, average="weighted", zero_division=0,
        )),
    }

---

## Section 7: Training Arguments

- `bf16=True` -- Blackwell sm_120 natively supports bfloat16
- `dataloader_num_workers=0` -- avoids spawn issues on Windows
- `report_to` -- optional W&B integration (plan Phase 6)
- `load_best_model_at_end=True` + `metric_for_best_model="macro_f1"`

In [17]:
# --- 7.1 Training arguments ---

from transformers import EarlyStoppingCallback

CKPT_DIR.mkdir(parents=True, exist_ok=True)
TRAINER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

epochs = QUICK_TEST_EPOCHS if QUICK_TEST else NUM_EPOCHS
steps_per_epoch = max(len(tokenized["train"]) // BATCH_SIZE, 1)
total_steps = steps_per_epoch * epochs
warmup_steps = int(WARMUP_RATIO * total_steps)

training_args = TrainingArguments(
    output_dir=str(TRAINER_OUTPUT_DIR),
    num_train_epochs=epochs,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps,
    lr_scheduler_type="cosine_with_min_lr",
    lr_scheduler_kwargs={"min_lr_rate": 0.1},
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=EVAL_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model=METRIC_FOR_BEST,
    greater_is_better=True,
    bf16=(device == "cuda"),
    fp16=False,
    dataloader_num_workers=0,
    report_to=["wandb"] if USE_WANDB else [],
    seed=SEED,
)

trainer = WeightedBCETrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    pos_weight=pos_weight,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
        ),
    ],
)

print(f"Trainer ready. Train steps ~{total_steps:,} "
      f"(warmup: {warmup_steps}, eval every {EVAL_STEPS})")

Trainer ready. Train steps ~1,280 (warmup: 76, eval every 100)


---

## Section 8: Train

Wall-time on RTX 5090 with `bf16`, batch 64, seq 256, 5 epochs +
`EarlyStoppingCallback(patience=3)`: **~12-15 minutes**.

`load_best_model_at_end=True` + `metric_for_best_model="macro_f1"`
ensures the loaded checkpoint is the macro-F1 peak.

In [18]:
# --- 8.1 Train ---

train_result = trainer.train()

print("\nTraining done.")
print(f"  Total train time:  {train_result.metrics['train_runtime']:.1f} s")
print(f"  Final train loss:  {train_result.metrics['train_loss']:.4f}")
print(f"  Steps/sec:         {train_result.metrics['train_steps_per_second']:.2f}")

Step,Training Loss,Validation Loss,Micro F1,Macro F1,Weighted F1
100,0.596494,0.504844,0.502344,0.210524,0.445218
200,0.508890,0.418502,0.639050,0.383161,0.613281
300,0.410219,0.340355,0.741732,0.492299,0.742330
400,0.345955,0.282975,0.820168,0.575583,0.811238
500,0.298471,0.259798,0.825598,0.591562,0.840061
600,0.267902,0.220890,0.851993,0.671909,0.861105
700,0.240597,0.202069,0.863979,0.706449,0.875456
800,0.217053,0.203486,0.849373,0.695286,0.867104
900,0.210218,0.187135,0.864584,0.717413,0.878088
1000,0.198382,0.185519,0.866538,0.709057,0.883283


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training done.
  Total train time:  635.2 s
  Final train loss:  0.3195
  Steps/sec:         2.02


---

## Section 9: Per-Label Threshold Tuning

The 0.5 threshold used during training is wrong for rare labels.
For each label, we find the F1-optimal threshold on validation
using `precision_recall_curve`.

In [19]:
# Sanity check: confirm the best checkpoint was loaded
print(f"trainer.state.best_metric: {trainer.state.best_metric:.4f}")
print(f"trainer.state.best_model_checkpoint: "
      f"{trainer.state.best_model_checkpoint}")

trainer.state.best_metric: 0.7174
trainer.state.best_model_checkpoint: checkpoints\_trainer_runs\checkpoint-900


In [20]:
# --- 9.1 Tune per-label thresholds on validation ---


def predict_probs(
    trainer: Trainer, dataset,
) -> tuple[np.ndarray, np.ndarray]:
    """Run inference. Returns (sigmoid_probs, label_ids)."""
    output = trainer.predict(dataset)
    logits = output.predictions
    probs = 1.0 / (1.0 + np.exp(-logits))
    return probs, output.label_ids


def tune_thresholds(
    probs: np.ndarray,
    labels: np.ndarray,
    label_names: list[str],
    *,
    fallback: float = 0.5,
) -> tuple[dict[str, float], dict[str, float]]:
    """Per-label F1-optimal threshold via PR curve."""
    thresholds: dict[str, float] = {}
    f1_at_best: dict[str, float] = {}
    for i, lbl in enumerate(label_names):
        y_true = labels[:, i].astype(int)
        y_prob = probs[:, i]
        if y_true.sum() == 0:
            thresholds[lbl] = fallback
            f1_at_best[lbl] = 0.0
            continue
        pr, rc, th = precision_recall_curve(y_true, y_prob)
        denom = pr[:-1] + rc[:-1]
        f1 = np.where(
            denom > 0,
            2 * pr[:-1] * rc[:-1] / np.where(denom > 0, denom, 1),
            0.0,
        )
        if len(f1) == 0:
            thresholds[lbl] = fallback
            f1_at_best[lbl] = 0.0
        else:
            best = int(np.argmax(f1))
            thresholds[lbl] = float(th[best])
            f1_at_best[lbl] = float(f1[best])
    return thresholds, f1_at_best


val_probs, val_labels = predict_probs(trainer, tokenized["validation"])
thresholds, val_f1_per_label = tune_thresholds(
    val_probs, val_labels, ALL_LABELS,
)

print(f"  {'label':<26s} {'threshold':>10s} {'val_f1':>8s}")
print(f"  {'-' * 26} {'-' * 10} {'-' * 8}")
for lbl in ALL_LABELS:
    print(f"  {lbl:<26s} {thresholds[lbl]:>10.3f} "
          f"{val_f1_per_label[lbl]:>8.3f}")

  label                       threshold   val_f1
  -------------------------- ---------- --------
  cleanliness_positive            0.613    0.974
  cleanliness_negative            0.613    0.781
  room_positive                   0.715    0.912
  room_negative                   0.751    0.756
  staff_positive                  0.622    0.930
  staff_negative                  0.736    0.722
  location_positive               0.739    0.918
  location_negative               0.742    0.681
  noise_positive                  0.657    0.974
  noise_negative                  0.662    0.926
  wifi_positive                   0.856    0.966
  wifi_negative                   0.576    0.375
  value_positive                  0.783    0.902
  value_negative                  0.819    0.714
  breakfast_positive              0.859    0.705
  breakfast_negative              0.684    0.483


---

## Section 10: Test Set Evaluation

Apply tuned thresholds to test set predictions. The test set is
from 20 held-out hotels that never appeared in training.

In [21]:
# --- 10.1 Test set evaluation with tuned thresholds ---


def apply_thresholds(
    probs: np.ndarray,
    thresholds: dict[str, float],
    label_names: list[str],
) -> np.ndarray:
    """Threshold per-label probs; return int array."""
    out = np.zeros_like(probs, dtype=int)
    for i, lbl in enumerate(label_names):
        out[:, i] = (probs[:, i] >= thresholds[lbl]).astype(int)
    return out


test_probs, test_labels = predict_probs(trainer, tokenized["test"])
test_preds = apply_thresholds(test_probs, thresholds, ALL_LABELS)
test_labels_int = test_labels.astype(int)

overall = {
    "micro_f1": float(f1_score(
        test_labels_int, test_preds, average="micro", zero_division=0,
    )),
    "macro_f1": float(f1_score(
        test_labels_int, test_preds, average="macro", zero_division=0,
    )),
    "weighted_f1": float(f1_score(
        test_labels_int, test_preds, average="weighted", zero_division=0,
    )),
}

print("=== TEST SET METRICS (tuned per-label thresholds) ===")
for k, v in overall.items():
    print(f"  {k:<14s} {v:.4f}")

=== TEST SET METRICS (tuned per-label thresholds) ===
  micro_f1       0.9004
  macro_f1       0.7451
  weighted_f1    0.9030


In [22]:
# --- 10.2 Per-label test report ---

per_label_test: dict[str, dict[str, float]] = {}
print(f"  {'label':<26s} {'support':>7s} {'P':>6s} {'R':>6s} "
      f"{'F1':>6s} {'thresh':>7s}")
print(f"  {'-' * 26} {'-' * 7} {'-' * 6} {'-' * 6} "
      f"{'-' * 6} {'-' * 7}")
for i, lbl in enumerate(ALL_LABELS):
    y_true = test_labels_int[:, i]
    y_pred = test_preds[:, i]
    support = int(y_true.sum())
    p = float(precision_score(y_true, y_pred, zero_division=0))
    r = float(recall_score(y_true, y_pred, zero_division=0))
    f1 = float(f1_score(y_true, y_pred, zero_division=0))
    per_label_test[lbl] = {
        "support": support, "precision": p, "recall": r,
        "f1": f1, "threshold": thresholds[lbl],
    }
    print(f"  {lbl:<26s} {support:>7,} {p:>6.3f} {r:>6.3f} "
          f"{f1:>6.3f} {thresholds[lbl]:>7.3f}")

LOW_SUPPORT_THRESHOLD = 30
low_support = [
    lbl for lbl, m in per_label_test.items()
    if m["support"] < LOW_SUPPORT_THRESHOLD
]
if low_support:
    print(f"\nLabels with test support < {LOW_SUPPORT_THRESHOLD} "
          f"(F1 unreliable): {low_support}")

  label                      support      P      R     F1  thresh
  -------------------------- ------- ------ ------ ------ -------
  cleanliness_positive           782  0.979  0.959  0.969   0.613
  cleanliness_negative           114  0.774  0.781  0.777   0.613
  room_positive                  746  0.933  0.890  0.911   0.715
  room_negative                  208  0.760  0.702  0.730   0.751
  staff_positive                 938  0.914  0.939  0.926   0.622
  staff_negative                  77  0.528  0.857  0.653   0.736
  location_positive              650  0.897  0.965  0.930   0.739
  location_negative               27  0.611  0.407  0.489   0.742
  noise_positive                 283  0.977  0.919  0.947   0.657
  noise_negative                 179  0.928  0.933  0.930   0.662
  wifi_positive                   56  0.946  0.946  0.946   0.856
  wifi_negative                    3  0.250  0.667  0.364   0.576
  value_positive                 204  0.897  0.941  0.919   0.783
  value_ne

In [23]:
# Macro-F1 excluding labels with insufficient test support
reliable_labels = [
    lbl for lbl in ALL_LABELS if lbl not in low_support
]
reliable_idx = [ALL_LABELS.index(lbl) for lbl in reliable_labels]

reliable_macro_f1 = f1_score(
    test_labels_int[:, reliable_idx],
    test_preds[:, reliable_idx],
    average="macro", zero_division=0,
)

print(f"Macro-F1 over {len(reliable_labels)} reliable labels: "
      f"{reliable_macro_f1:.4f}")
print(f"Macro-F1 over all 16 labels: "
      f"{overall['macro_f1']:.4f}")
print(f"Excluded (test support < 30): {low_support}")

Macro-F1 over 13 reliable labels: 0.8515
Macro-F1 over all 16 labels: 0.7451
Excluded (test support < 30): ['location_negative', 'wifi_negative', 'breakfast_negative']


---

## Section 11: Persist Artifacts

Save the trained model + tokenizer (HuggingFace standard layout)
plus custom artifacts:
- `thresholds.json` -- consumed by NB04 at inference time
- `label_names.json` -- ordered list of 16 labels
- `metrics.json` -- val + test metrics for the report
- `training_log.json` -- step-wise loss history

In [24]:
# --- 11.1 Persist model + thresholds + metrics ---

trainer.save_model(str(CKPT_DIR))
tokenizer.save_pretrained(str(CKPT_DIR))

with (CKPT_DIR / "thresholds.json").open("w") as f:
    json.dump(thresholds, f, indent=2)

with (CKPT_DIR / "label_names.json").open("w") as f:
    json.dump(ALL_LABELS, f, indent=2)

metrics_payload = {
    "model_name": MODEL_NAME,
    "clf_base_model": CLF_BASE_MODEL,
    "mlm_adapted": RUN_MLM_ADAPTATION,
    "max_seq_length": MAX_SEQ_LENGTH,
    "num_epochs": epochs,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "pos_weight_strategy": POS_WEIGHT_STRATEGY,
    "pos_weight_cap": POS_WEIGHT_CAP,
    "pos_weight_per_label": dict(zip(
        ALL_LABELS, pos_weight.tolist(), strict=True,
    )),
    "val_f1_per_label": val_f1_per_label,
    "test_overall": overall,
    "test_per_label": per_label_test,
    "low_support_labels": low_support,
    "splits": {
        name: int(len(sdf)) for name, sdf in split_dfs.items()
    },
    "quick_test": QUICK_TEST,
}
with (CKPT_DIR / "metrics.json").open("w") as f:
    json.dump(metrics_payload, f, indent=2)

log_history = [
    {k: float(v) if isinstance(v, (np.floating,)) else v
     for k, v in entry.items()}
    for entry in trainer.state.log_history
]
with (CKPT_DIR / "training_log.json").open("w") as f:
    json.dump(log_history, f, indent=2)

print("Saved:")
for path in sorted(CKPT_DIR.iterdir()):
    if path.is_file():
        size_mb = path.stat().st_size / 1e6
        print(f"  {path}  ({size_mb:.2f} MB)")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved:
  checkpoints\hotel_curator_clf\config.json  (0.00 MB)
  checkpoints\hotel_curator_clf\label_names.json  (0.00 MB)
  checkpoints\hotel_curator_clf\metrics.json  (0.01 MB)
  checkpoints\hotel_curator_clf\model.safetensors  (498.66 MB)
  checkpoints\hotel_curator_clf\thresholds.json  (0.00 MB)
  checkpoints\hotel_curator_clf\tokenizer.json  (3.56 MB)
  checkpoints\hotel_curator_clf\tokenizer_config.json  (0.00 MB)
  checkpoints\hotel_curator_clf\training_args.bin  (0.01 MB)
  checkpoints\hotel_curator_clf\training_log.json  (0.01 MB)


---

## Summary

| Artifact | Path | Description |
|---|---|---|
| MLM checkpoint | `checkpoints/roberta-tripadvisor-mlm/` | Domain-adapted RoBERTa (Phase 5) |
| Model + tokenizer | `checkpoints/hotel_curator_clf/` | HF-format checkpoint |
| Thresholds | `checkpoints/hotel_curator_clf/thresholds.json` | Per-label decision thresholds (NB04 input) |
| Label names | `checkpoints/hotel_curator_clf/label_names.json` | Index -> label name |
| Metrics | `checkpoints/hotel_curator_clf/metrics.json` | Hyperparams + val/test metrics |
| Training log | `checkpoints/hotel_curator_clf/training_log.json` | Step-wise loss + eval F1 history |

### Decisions documented

| Decision | Rationale |
|---|---|
| **MLM domain adaptation (3 epoch)** | Phase 5 of ML Implementation Plan; adapts embeddings to hospitality jargon. Expected +2-3 ppts macro-F1 |
| `max_seq_length=256` over spec's 128 | Corpus p95 is 275 words; 128 truncates ~50% of content |
| `pos_weight` with **sqrt** strategy (capped at 50x) | Hard cap loses rank ordering between rare labels; sqrt preserves it |
| `BATCH_SIZE = 64` | RTX 5090 has VRAM headroom; larger batch stabilizes gradients with high `pos_weight` |
| `NUM_EPOCHS = 5` with `EarlyStoppingCallback(patience=3)` | 5 epochs let rare-label heads converge fully |
| `cosine_with_min_lr` (10% floor) | Keeps late steps learning rare-label features |
| `metric_for_best_model="macro_f1"` | Treats all 16 labels equally |
| Per-label threshold tuning on val | 0.5 threshold is wrong when prevalences span 0.34% to 47.5% |
| Optional W&B logging (`USE_WANDB`) | Plan specifies W&B; made opt-in to avoid hard dependency |

### ML Implementation Plan coverage

| Plan Phase | Status |
|---|---|
| **Phase 5: MLM domain adaptation** | Implemented (Section 2) |
| **Phase 6: Multi-label classification** | Implemented (Sections 5-10) |
| **Phase 6: W&B logging** | Implemented as opt-in (`USE_WANDB` flag) |
| **Phase 7: Attribute post-processing** | Not applicable in NB03 -- Room Size regex is in NB02a; Location/Safety require external data not in this dataset. Cascading inference logic belongs in NB04. |

### Next: Notebook 04 -- Aggregation, Tier Mapping, and Hotel-Level Eval

NB04 will:
1. Load the checkpoint + `thresholds.json` + label names
2. Run inference on all 20,475 reviews
3. Aggregate to per-hotel per-attribute signals
4. Apply tier mapping with critical-attribute floors
5. Quantitative test-split evaluation on 20 held-out hotels
6. Top-3 evidence reviews per (hotel, attribute, polarity)
7. 20-hotel stratified audit table
8. Optional gold-set evaluation